# Notebook 01 — Data Preprocessing

**Goal**: Load and clean both datasets, align timestamps, and export the processed tweet and match-event tables that all downstream notebooks depend on.

Sections:
1. Environment setup
2. Load raw datasets
3. Clean and filter tweets
4. Inspect match events
5. Build pressure-window aggregates
6. Export processed files

In [ ]:
# ── 1. Environment setup ──────────────────────────────────────────────────
# If running on Google Colab, mount Drive and clone the repo first:
#
# from google.colab import drive
# drive.mount('/content/drive')
# !git clone https://github.com/sjekic/nlp-group-project.git
# %cd nlp-group-project
# !pip install -r requirements.txt -q

import sys, os

# Robust path: works whether notebook is run from repo root, notebooks/, or Colab
_nb_dir = os.path.dirname(os.path.abspath(globals().get("__file__", os.getcwd())))
_repo_root = _nb_dir if os.path.isdir(os.path.join(_nb_dir, "src")) else os.path.dirname(_nb_dir)
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.preprocessing import load_tweets, load_match_events, build_pressure_windows

# ── Paths — change DATA_DIR to your local or Drive path ──────────────────
DATA_DIR    = "../data"
OUTPUT_DIR  = "../outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TWEET_PATH  = f"{DATA_DIR}/premier_league_twitter_comments_match_windows_2020_07_09_to_2020_10_13.csv"
MATCH_PATH  = f"{DATA_DIR}/premier_league_combined_dataset_2020.csv"

In [ ]:
# ── 2. Load raw datasets ──────────────────────────────────────────────────
tweets_raw = pd.read_csv(TWEET_PATH)
match_raw  = pd.read_csv(MATCH_PATH)

print(f"Tweets raw shape : {tweets_raw.shape}")
print(f"Match  raw shape : {match_raw.shape}")
tweets_raw.head(3)

In [ ]:
# ── 3. Clean and filter tweets ────────────────────────────────────────────
tweets = load_tweets(TWEET_PATH)

print(f"After cleaning: {len(tweets):,} tweets, {tweets['fixture_id'].nunique()} matches")
print(f"Minute range  : {tweets['relative_minute_from_kickoff'].min():.0f} – {tweets['relative_minute_from_kickoff'].max():.0f}")
print(f"Polarity range: {tweets['polarity'].min():.3f} – {tweets['polarity'].max():.3f}")
tweets[["fixture_id","window_5min","text_clean","polarity"]].head(5)

In [ ]:
# Tweet count distribution per match
tweet_counts = tweets.groupby("match")["text_clean"].count().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 4))
tweet_counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Tweet count per match (after filtering)", fontsize=12, fontweight="bold")
ax.set_ylabel("Number of tweets")
ax.set_xlabel("")
ax.tick_params(axis="x", labelsize=6, rotation=90)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/tweet_counts_per_match.png", dpi=150)
plt.show()

In [ ]:
# Tweet volume over match minutes (aggregated across all matches)
minute_vol = tweets.groupby("window_5min").size()

fig, ax = plt.subplots(figsize=(12, 4))
minute_vol.plot(kind="bar", ax=ax, color="#e76f51", edgecolor="white")
ax.set_title("Tweet volume by 5-minute window (all matches)", fontsize=12, fontweight="bold")
ax.set_xlabel("Match minute")
ax.set_ylabel("Tweet count")
ax.tick_params(axis="x", labelsize=8)
ax.axvline(9, color="grey", linewidth=2, linestyle="--", label="Kick-off (min 0)")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/tweet_volume_by_minute.png", dpi=150)
plt.show()

In [ ]:
# ── 4. Inspect match events ───────────────────────────────────────────────
match_events = load_match_events(MATCH_PATH)

print(f"Match events shape : {match_events.shape}")
print(f"Fixtures           : {match_events['fixture_id'].nunique()}")
print("\nEvent type counts:")
print(match_events['event_type_name'].value_counts().to_string())

In [ ]:
# Goals per match distribution
goals = match_events[match_events["event_type_name"] == "Goal"]
goals_per_match = goals.groupby("fixture_id").size()

fig, ax = plt.subplots(figsize=(8, 4))
goals_per_match.value_counts().sort_index().plot(kind="bar", ax=ax, color="#2a9d8f", edgecolor="white")
ax.set_title("Goals per match distribution (2024/25)", fontsize=12, fontweight="bold")
ax.set_xlabel("Number of goals")
ax.set_ylabel("Number of matches")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/goals_per_match.png", dpi=150)
plt.show()

In [ ]:
# ── 5. Build pressure-window aggregates ──────────────────────────────────
pressure_windows = build_pressure_windows(match_events, window_size=5)

print(f"Pressure windows shape: {pressure_windows.shape}")
pressure_windows.describe()

In [ ]:
# Pressure distribution over match minutes (all matches)
pressure_by_min = pressure_windows.groupby("window_5min")["mean_pressure"].mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(pressure_by_min.index, pressure_by_min.values, alpha=0.5, color="#e63946")
ax.plot(pressure_by_min.index, pressure_by_min.values, color="#e63946", linewidth=2)
ax.axvline(45, color="grey", linestyle="--", linewidth=1.5, label="Half-time")
ax.set_title("Average match pressure by 5-minute window (all 2024/25 matches)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Match minute")
ax.set_ylabel("Mean pressure value")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/pressure_by_minute.png", dpi=150)
plt.show()

In [ ]:
# ── 6. Export processed files ─────────────────────────────────────────────
tweets.to_csv(f"{OUTPUT_DIR}/tweets_cleaned.csv", index=False)
match_events.to_csv(f"{OUTPUT_DIR}/match_events_cleaned.csv", index=False)
pressure_windows.to_csv(f"{OUTPUT_DIR}/pressure_windows.csv", index=False)

print("Exported:")
print(f"  {OUTPUT_DIR}/tweets_cleaned.csv")
print(f"  {OUTPUT_DIR}/match_events_cleaned.csv")
print(f"  {OUTPUT_DIR}/pressure_windows.csv")